# Train gpt-oss from scratch with NeonEngine on a T4

Runs NeonEngine's own Vulkan/Slang trainer (`NeonML --gpt-oss-train`), not PyTorch.

**First time:** on your PC run `python train/colab/pack_for_colab.py`, upload the zip to Google Drive as
`MyDrive/neon/NeonEngine_colab.zip`, then select **Runtime > Change runtime type > T4 GPU** and run all cells.

**After a disconnect / usage limit:** reconnect a T4 runtime and **run all cells again**. Training resumes
from the newest checkpoint on Drive (`MyDrive/neon/run/latest.nckpt`, written every 30 minutes and when you
interrupt the training cell).

In [ ]:
# 1. Google Drive holds everything that must survive the session: code zip, tokenized data, checkpoints.
from google.colab import drive
drive.mount('/content/drive')
import os
NEON = '/content/drive/MyDrive/neon'
os.makedirs(NEON, exist_ok=True)
os.environ['NEON'] = NEON

In [ ]:
# 2. Code: unzip the package from Drive (or uncomment the git clone line once your changes are pushed).
!rm -rf /content/src && mkdir -p /content/src && unzip -oq "$NEON/NeonEngine_colab.zip" -d /content/src
# !git clone --depth 1 https://github.com/DaRealNeonCoder/NeonEngine /content/repo && ln -s /content/repo/NeonEngine /content/src/NeonEngine
!ls /content/src/NeonEngine

In [ ]:
# 3. Vulkan SDK + NVIDIA Vulkan driver + build NeonML (about 3-5 minutes per session).
!bash /content/src/NeonEngine/train/colab/setup_colab.sh /content/src/NeonEngine

In [ ]:
# 4. Training data: TinyStories with an 8192-token BPE tokenizer trained on it (~460M tokens; as compact as
#    GPT-2's 50k vocabulary on this text, with a 6x smaller LM head). Built once (~15 min) and kept on Drive;
#    later sessions copy it to the local disk (random reads from Drive are slow).
!pip -q install tiktoken datasets
import os, shutil
local, saved = '/content/data/tinystories_bpe8k', f'{NEON}/data/tinystories_bpe8k'
if not os.path.exists(f'{saved}/meta.json'):
    !python /content/src/NeonEngine/train/prepare_data.py --out {local} --vocab-size 8192
    shutil.copytree(local, saved, dirs_exist_ok=True)
elif not os.path.exists(f'{local}/meta.json'):
    shutil.copytree(saved, local, dirs_exist_ok=True)
!cat {local}/meta.json

## 5. Train

Model: `train/configs/t4.json` (gpt-oss architecture: 8 layers, hidden 512, 16 experts top-4, ~114M
parameters with the 8k vocabulary). Each step is 16 x 512 tokens. Checkpoints go to `/content/run` and are
mirrored to `MyDrive/neon/run`. Interrupting this cell (stop button) saves a checkpoint before exiting.
Rerunning it continues where it stopped. When it reaches `--steps` it writes the inference checkpoint to
`MyDrive/neon/run/export`.

GEMMs run on the T4's fp16 tensor cores with dynamic loss scaling (`--precision fp16`, the default when
available; `fp32` is the old path). The LR schedule is warmup-stable-decay (`--schedule wsd`): constant LR, then a
linear cooldown over the last 20% of `--steps`, so `--steps` can still be raised before the cooldown starts.
The log shows validation loss in bits per byte, comparable across tokenizers. To see where GPU time goes, add
`--profile 5` (prints a per-kernel table after 5 steps and exits).

In [ ]:
!source /content/neon_env.sh && $NEON_ML --gpt-oss-train train     --config $NEON_SRC/train/configs/t4.json --data /content/data/tinystories_bpe8k     --out /content/run --mirror "$NEON/run" --shaders $NEON_SRC/shaders/ML     --batch 16 --seq 512 --steps 30000 --warmup 500 --lr 1e-3 --min-lr 1e-4 --schedule wsd     --eval-every 500 --ckpt-minutes 30

## 6. Generate text with NeonEngine inference

Uses the exported checkpoint (or exports the newest training checkpoint first).

In [ ]:
import subprocess
PROMPT = "Once upon a time, there was a little dragon who"
!source /content/neon_env.sh && test -f "$NEON/run/export/config.json" || $NEON_ML --gpt-oss-train export --ckpt "$NEON/run" --out "$NEON/run/export"
eot = subprocess.check_output(['python', '/content/src/NeonEngine/shaders/ML/tests/gpt_oss_tokens.py', '--model', f'{NEON}/run/export', 'eot'], text=True).strip()
ids = subprocess.check_output(['python', '/content/src/NeonEngine/shaders/ML/tests/gpt_oss_tokens.py', '--model', f'{NEON}/run/export', 'encode', PROMPT], text=True).strip()
out = subprocess.run(f'source /content/neon_env.sh && $NEON_ML --gpt-oss "$NEON/run/export" --tokens {ids} --max-new 200 '
                     f'--temperature 0.8 --top-k 50 --stop {eot} --quiet --shaders $NEON_SRC/shaders/ML',
                     shell=True, executable='/bin/bash', capture_output=True, text=True)
gen = [l for l in out.stdout.splitlines() if l and l[0].isdigit() and ',' in l]
if not gen:
    print(out.stdout, out.stderr)
else:
    print(subprocess.check_output(['python', '/content/src/NeonEngine/shaders/ML/tests/gpt_oss_tokens.py', '--model', f'{NEON}/run/export', 'decode', ids + ',' + gen[-1]], text=True))